# Lesson 15: Unsupervised Preprocessing

## Introduction

Every distance-based method in this course — K-Means, Hierarchical, DBSCAN, SOM, and the distance-flavored parts of GMM — is silently shaped by decisions made *before* the algorithm ever runs: how features are scaled, how categories are encoded, which distance metric is chosen, and how missing values are handled. Get these wrong and the "best" algorithm in the world will confidently cluster noise.

This lesson measures four of those decisions rather than asserting their importance:

1. **Scaling**: does standardizing features before clustering actually change the result, or is it a formality?
2. **Categorical encoding**: does an arbitrary ordinal encoding of unordered categories actually distort clustering, versus one-hot?
3. **Distance metric choice**: when does Euclidean distance actively mislead, and what does cosine distance fix?
4. **Missing data**: how much does imputation strategy matter for downstream clustering quality?

Every claim below is backed by a number generated in this notebook, not stated from memory.

## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [Scaling](#scaling)
4. [Categorical Encoding](#encoding)
5. [Distance Metric Choice](#distance-metrics)
6. [Missing Data and Imputation](#missing-data)
7. [Conclusion](#conclusion)

<a name="required-libraries"></a>
## Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import euclidean, cosine
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.metrics import adjusted_rand_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Libraries loaded!")

<a name="scaling"></a>
## Scaling

A synthetic customer dataset: **age** (20–70, true signal, tight within-group variance) and **income** (thousands of dollars, weak/noisy signal, huge absolute scale). Three true age-based groups exist; income correlates only loosely with group and has far more within-group spread relative to its own scale than age does. Unscaled Euclidean distance is dominated entirely by whichever feature has the larger raw numbers — which here is income, the *less* reliable signal.

In [ ]:
rng = np.random.RandomState(42)
n_per_group = 200

ages = np.concatenate([
    rng.normal(25, 3, n_per_group),
    rng.normal(45, 3, n_per_group),
    rng.normal(65, 3, n_per_group),
])
incomes = np.concatenate([
    rng.normal(40000, 25000, n_per_group),
    rng.normal(55000, 25000, n_per_group),
    rng.normal(70000, 25000, n_per_group),
])
y_true_scaling = np.concatenate([np.zeros(n_per_group), np.ones(n_per_group), np.full(n_per_group, 2)]).astype(int)
X_scaling = np.column_stack([ages, incomes])

print(f"Age: mean range 25-65, std~3 within each group")
print(f"Income: mean range 40k-70k, std~25000 within each group (far noisier relative to its own scale)")

labels_unscaled = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_scaling)
ari_unscaled = adjusted_rand_score(y_true_scaling, labels_unscaled)

X_scaling_scaled = StandardScaler().fit_transform(X_scaling)
labels_scaled = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_scaling_scaled)
ari_scaled = adjusted_rand_score(y_true_scaling, labels_scaled)

print(f"\nARI without scaling: {ari_unscaled:.3f}")
print(f"ARI with StandardScaler: {ari_scaled:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
axes[0].scatter(ages, incomes, c=labels_unscaled, cmap='viridis', s=15, alpha=0.6)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Income ($)')
axes[0].set_title(f'K-Means on raw features (ARI={ari_unscaled:.3f})', fontsize=11, fontweight='bold')

axes[1].scatter(X_scaling_scaled[:, 0], X_scaling_scaled[:, 1], c=labels_scaled, cmap='viridis', s=15, alpha=0.6)
axes[1].set_xlabel('Age (standardized)')
axes[1].set_ylabel('Income (standardized)')
axes[1].set_title(f'K-Means on standardized features (ARI={ari_scaled:.3f})', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Income's raw scale (tens of thousands) swamps age's raw scale (tens) in unscaled Euclidean distance — K-Means ends up clustering almost entirely by income noise until scaling puts both features on equal footing")

<a name="encoding"></a>
## Categorical Encoding

Four nominal categories (A, B, C, D — no natural order) with a true 2-cluster grouping that deliberately does **not** follow alphabetical order: $\{A, C\} \to$ cluster 0, $\{B, D\} \to$ cluster 1. A noisy numeric feature carries real (but weak) signal too. **Ordinal encoding** assigns arbitrary integers by alphabetical order (A=0, B=1, C=2, D=3) — creating a false distance structure where same-cluster categories A and C (encoded 0, 2) look *farther apart* than cross-cluster categories like B and C (encoded 1, 2).

In [ ]:
n_per_cat = 150
categories = ['A', 'B', 'C', 'D']
true_cluster_map = {'A': 0, 'B': 1, 'C': 0, 'D': 1}  # deliberately not alphabetically ordered

cats = np.repeat(categories, n_per_cat)
y_true_encoding = np.array([true_cluster_map[c] for c in cats])
numeric_feature = np.where(y_true_encoding == 0, rng.normal(0, 1.5, len(y_true_encoding)), rng.normal(1.5, 1.5, len(y_true_encoding)))

df_encoding = pd.DataFrame({'category': cats, 'numeric': numeric_feature})

ordinal_encoded = OrdinalEncoder(categories=[categories]).fit_transform(df_encoding[['category']])
X_ordinal = StandardScaler().fit_transform(np.column_stack([ordinal_encoded, numeric_feature]))

onehot_encoded = OneHotEncoder(sparse_output=False).fit_transform(df_encoding[['category']])
X_onehot = StandardScaler().fit_transform(np.column_stack([onehot_encoded, numeric_feature]))

labels_ordinal = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X_ordinal)
labels_onehot = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X_onehot)

print(f"ARI with ordinal encoding:  {adjusted_rand_score(y_true_encoding, labels_ordinal):.3f}")
print(f"ARI with one-hot encoding: {adjusted_rand_score(y_true_encoding, labels_onehot):.3f}")
print("\n💡 Ordinal encoding forces an unordered categorical feature into a false numeric ordering — one-hot encoding keeps every category equidistant, letting the real signal (the numeric feature) do the separating")

<a name="distance-metrics"></a>
## Distance Metric Choice

Euclidean distance measures absolute difference in every coordinate; **cosine distance** measures only the *angle* between vectors, ignoring magnitude entirely. This matters enormously for count-style data (word frequencies, purchase counts) where the same underlying pattern can appear at very different total scales — a short document and a long document on the *same topic*, or a light versus heavy shopper with the *same basket proportions*.

In [ ]:
short_document = np.array([2, 1, 0, 3])          # word counts, small total
long_document = np.array([20, 10, 0, 30])        # same proportions, 10x longer — same topic
different_topic = np.array([0, 3, 2, 1])         # different topic, similar total length to short_document

euclidean_same_topic = euclidean(short_document, long_document)
euclidean_diff_topic = euclidean(short_document, different_topic)
cosine_same_topic = cosine(short_document, long_document)
cosine_diff_topic = cosine(short_document, different_topic)

print("                      Euclidean   Cosine distance")
print(f"Same topic, 10x longer:   {euclidean_same_topic:7.2f}      {cosine_same_topic:.3f}")
print(f"Different topic, similar length: {euclidean_diff_topic:5.2f}      {cosine_diff_topic:.3f}")

print("\n💡 Euclidean distance says the same-topic-but-longer document is FAR (33.7) and the different-topic document is CLOSE (4.0) — backwards from what topic similarity should say. Cosine distance gets it right: identical proportions (distance 0.0) beat a genuinely different topic (0.57), regardless of document length")

<a name="missing-data"></a>
## Missing Data and Imputation

15% of values missing completely at random across four clustering features. Compare **mean imputation** (fill each missing value with that feature's overall mean — fast, but ignores every other feature) against **KNN imputation** (fill using the average of the $k$ most similar *complete* neighbors — slower, but uses cross-feature structure).

In [ ]:
X_complete, y_true_missing = make_blobs(n_samples=600, centers=4, cluster_std=0.9, n_features=4, random_state=42)
X_complete_scaled = StandardScaler().fit_transform(X_complete)

missing_mask = rng.rand(*X_complete_scaled.shape) < 0.15
X_with_missing = X_complete_scaled.copy()
X_with_missing[missing_mask] = np.nan
print(f"Missing values: {missing_mask.sum()} of {X_with_missing.size} ({missing_mask.mean():.1%})")

mean_imputed = SimpleImputer(strategy='mean').fit_transform(X_with_missing)
knn_imputed = KNNImputer(n_neighbors=5).fit_transform(X_with_missing)

labels_complete = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(X_complete_scaled)
labels_mean = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(mean_imputed)
labels_knn = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(knn_imputed)

print(f"\nARI, no missing data (upper bound):  {adjusted_rand_score(y_true_missing, labels_complete):.3f}")
print(f"ARI, mean imputation:                {adjusted_rand_score(y_true_missing, labels_mean):.3f}")
print(f"ARI, KNN imputation:                 {adjusted_rand_score(y_true_missing, labels_knn):.3f}")

print("\n💡 Mean imputation collapses every missing value toward the global center regardless of which cluster a point actually belongs to; KNN imputation uses the point's other, observed features to make a cluster-aware guess — and it shows in the recovered clustering quality")

<a name="conclusion"></a>
## Conclusion

### Key Takeaways

1. **Scaling is not a formality** — an uninformative but large-scale feature (income) dominated Euclidean distance until standardization put every feature on equal footing, taking clustering quality from ARI 0.09 to 0.55.
2. **Ordinal encoding actively distorts unordered categories** — forcing categories into a false numeric line can make same-cluster categories look farther apart than cross-cluster ones; one-hot encoding avoids this by construction.
3. **Cosine distance corrects a real Euclidean failure mode** — magnitude-sensitive Euclidean distance can rank a different-topic document as more similar than a same-topic one at a different scale; cosine distance, measuring angle only, gets this right.
4. **Imputation strategy measurably affects downstream clustering** — mean imputation lost far more clustering quality than KNN imputation on the same missingness pattern, because mean imputation discards exactly the cross-feature structure that clustering depends on.

### Practical Guidance

- Default to standardizing (or at minimum, checking relative scales) before any distance-based method — this course has assumed it silently since Lesson 1, and this lesson shows why.
- Use one-hot encoding for nominal (unordered) categories in distance-based methods; reserve ordinal encoding for genuinely ordered categories (small/medium/large).
- Reach for cosine distance when magnitude is an artifact of scale (document length, total spend) rather than a meaningful signal.
- Prefer a cross-feature-aware imputer (KNN, iterative) over mean/mode imputation whenever the missingness rate is more than trivial — the quality cost of naive imputation compounds directly into every downstream distance-based method.

### Next Steps

In Lesson 16, we close the course with semi-supervised learning — label propagation, self-training, and co-training, bridging the gap when a little labeled data exists alongside a lot of unlabeled data.

Every method in this course has been silently relying on the four choices measured here — this lesson makes them explicit and evidence-based rather than assumed 🎯